# Practice 2 · CIFAR-10 with a pretrained ResNet-18

**Interviewer:** "We fine-tune an ImageNet ResNet-18 on CIFAR-10: freeze the backbone and train only a new last layer. It crashes, and when it ran before, the reported accuracy looked too good to be true. Please walk me through it, tell me what you're checking, and fix it."

There are **8 problems**: 2 crash the code, the rest are silent (the code runs, but learns badly or reports wrong numbers).

**How to work**
1. Read the whole notebook first and say out loud what each part does.
2. Run the cells and fix crash errors one at a time. Read each error message: it tells you where to look.
3. Once it runs, go through it again for the silent bugs.

*Runtime → Change runtime type → T4 GPU.*

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

BATCH_SIZE = 128
EPOCHS = 3
LR = 1e-3
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # ImageNet stats

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

### Data

In [ ]:
train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
test_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

full_train = datasets.CIFAR10("/content/data", train=True, download=True, transform=train_tf)
test_ds = datasets.CIFAR10("/content/data", train=False, download=True, transform=test_tf)

n_val = 5000
train_ds, val_ds = random_split(full_train, [len(full_train) - n_val, n_val])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=2)
print("train:", len(train_ds), "val:", len(val_ds), "test:", len(test_ds))

### Model

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

model.fc = nn.Linear(model.fc.in_features, 10)
for p in model.parameters():
    p.requires_grad = False

model = model.to(device)
criterion = nn.CrossEntropyLoss()

### Train and evaluate functions

In [ ]:
def train_one_epoch(model, loader):
    model.train()
    total_loss, n = 0.0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.float().to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        n += images.size(0)
    return total_loss / n


def evaluate(model, loader):
    model.train()
    correct, total = 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    return correct / total

### Training

In [ ]:
best_acc, best_state = 0.0, None
for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader)
    acc = evaluate(model, test_loader)
    print(f"epoch {epoch}: train loss {train_loss:.3f} | acc {acc:.3f}")
    if acc > best_acc:
        best_acc = acc
        best_state = model.state_dict()

model.load_state_dict(best_state)
print("best accuracy:", best_acc)

### Your notes
> Bug: cell … → why it's wrong → my fix → crash or silent?